# Portfolio construction 1 of 3: what to hold

Notebooks: [1 hold](portfolio_construction_1_hold.ipynb) · [2 scale](portfolio_construction_2_scale.ipynb) · [3 trade](portfolio_construction_3_trade.ipynb), matching `src/portfolio_construction/{hold,scale,trade}`. In-sample, 1976-2014, DCC-GARCH risk model (the project's configured model, with the rescale), primary cost 2 bps per traded notional (`primary_cost_bps`); data conditional on the unverified March 1997 endpoint (`docs/DATA_STATUS.md`).

## Goal

Improve the model. Here: does inverse-volatility sizing leave risk concentrated, and would a better relative sizing improve risk-adjusted return? Judge every change at matched realised volatility, with Sharpe, turnover, cost, drawdown, class risk share and effective bets.

## Current pipeline

**What we are doing.** Choose which markets to hold, long or short, and how large each position is relative to the others. No overall scale yet.

**Inputs**
- Trend signal per market: sign of the lagged 12-month return sum (FX blends 3 and 12 months).
- Forecast annual volatility per market, from the risk model.
- Config: instrument scale 0.40, volatility floor 2%, minimum 10 eligible markets, excluded YM and EC.

**How we process them** (`hold/sizing.py`)
1. **Eligible** if the signal is known, volatility is positive and the market is not excluded. Fewer than 10 eligible markets: the book is flat.
2. **Size** each eligible market as `0.40 / (volatility × number eligible)`, with volatility floored at 2%.
3. **Sign** the size by the trend signal: long, short or flat.
4. *(Off by default)* **Class balance** rescales each asset class to an equal share of the risk ceiling (`hold/class_balance.py`).

*Example:* a market with 10% volatility, 37 eligible markets and a positive signal gets `0.40 / (0.10 × 37)` = 10.8% of capital, long. A market with 20% volatility gets half that.

**Output.** One signed weight per market, passed to the scale stage.

## Implications

- Sizing uses only each market's own volatility. The covariance enters later, as one scalar. The book is inverse-volatility with a global scale, not an optimisation.
- Correlated clusters get weight by headcount (22 commodity markets of about 37 active).
- No return forecast enters sizing, so no Markowitz error maximisation. Covariance instability only matters for edits that invert or use correlations.

## Results and interpretation

Euler risk contribution `rc_i = w_i (Qw)_i / w'Qw`. Effective bets = `1 / sum(rc_i²)`. Condition number of the active covariance.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]


from risk_model_study import load_study
from run_pc_scale import model_parts

from src.backtest.qps_baseline import forecasts_from

pd.options.display.float_format = "{:.3f}".format
study = load_study()
settings, base, returns, source, known, meta = study.inputs
parts = model_parts(study, "configured")
targets, covariance, _ = forecasts_from(parts, returns, meta, base)
labels = meta.AssetClass.reindex(returns.columns)
print(f"{returns.shape[1]} markets; trend target months: {len(targets['trend'])}")

58 markets; trend target months: 468


In [2]:
trend = targets["trend"]
rows = []
for month in trend.index:
    w = trend.loc[month].to_numpy()
    active = w != 0
    if active.sum() < base["minimum_instruments"]:
        continue
    Q = covariance[month]
    contribution = w * (Q @ w) / (w @ Q @ w)
    by_class = pd.Series(contribution, index=trend.columns).groupby(labels).sum()
    eig = np.linalg.eigvalsh(Q[np.ix_(active, active)])
    rows.append(
        {
            "month": month,
            "markets": int(active.sum()),
            "effective_bets": 1 / np.sum(contribution[active] ** 2),
            "condition_number": eig.max() / eig.min(),
            **by_class.add_prefix("risk_share_").to_dict(),
        }
    )
shape = pd.DataFrame(rows).set_index("month")
shape["distinct_cov_entries"] = shape.markets * (shape.markets + 1) / 2
shape["effective_bets_ratio"] = shape.effective_bets / shape.markets
shape.describe().T[["mean", "min", "50%", "max"]]

,mean,min,50%,max
markets,36.839,11.000,39.000,55.000
effective_bets,22.963,5.356,22.748,47.345
condition_number,3583.834,17.549,2604.893,33076.434
risk_share_COMM,0.576,0.088,0.543,1.000
risk_share_EQ,0.155,-0.022,0.075,0.643
risk_share_FI,0.157,-0.022,0.108,0.619
risk_share_FX,0.112,-0.024,0.097,0.430
distinct_cov_entries,788.981,66.000,780.000,1540.000
effective_bets_ratio,0.632,0.302,0.640,0.928


In [3]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
share = shape.filter(like="risk_share_").rename(columns=lambda c: c.replace("risk_share_", ""))
share.index = share.index.to_timestamp() if hasattr(share.index, "to_timestamp") else share.index
share.plot(ax=axes[0], title="Share of forecast risk by asset class")
axes[1].plot(share.index, shape.markets, label="active markets")
axes[1].plot(share.index, shape.effective_bets, label="effective bets (1/HHI of risk)")
axes[1].set_title("Diversification actually obtained"); axes[1].legend()
axes[2].semilogy(share.index, shape.condition_number)
axes[2].set_title("Condition number of active covariance")
plt.tight_layout(); plt.show()

/var/folders/2s/4bwc4_c534sg_8zdq7wkq00c0000gn/T/ipykernel_45494/118578350.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


- **Commodities carry about 58% of forecast risk**, with 22 of about 37 active markets.
- **Effective bets average about 23** against about 37 markets (63%), and fall to about 5.4 at the minimum.
- **Condition number averages about 3,600** (peak about 33,000): about 800 covariance entries from 36 months. Harmless to sizing today. It matters for HRP or mean-variance, and is about three times worse than under the rolling model.

## AFML improvements

Read: Ch. 16 (16.2-16.6), Ch. 14 (14.5.1, 14.7.3).

| Idea | What the book says | Use here |
|---|---|---|
| Markowitz's curse (16.3) | Correlated assets make the inverted covariance unstable; a non-singular `N×N` needs at least `N(N+1)/2` IID observations. | Condition number about 3,600 on 36 months: avoid mean-variance here. |
| Hierarchical Risk Parity (16.4) | Cluster by `sqrt((1-ρ)/2)`, quasi-diagonalise, split weight top-down by cluster variance. No inversion. | A data-driven version of class balance for steps 3-4. |
| Out-of-sample test (16.5-16.6) | Monte Carlo, 10 assets, 10,000 runs: out-of-sample variance 0.1157 (min-variance CLA), 0.0928 (inverse-variance), 0.0671 (HRP). | Judge edits out of sample. Simulated Gaussian data: a design hint only. |
| Bet concentration (14.5.1) | HHI-style concentration of bets. | Applied here to risk contributions. |
| Deflated Sharpe (14.7.3) | Judge a Sharpe against the maximum expected from the number of trials. | Every variant is a trial on an inspected window. |

## Suggested changes and experiments

| # | Change | Experiment | Effort |
|---|---|---|---|
| P0 | Add diagnostics to every run | Record class risk share, effective bets and trial count for the baseline as the reference. No change to `src/`. | Low |
| P1 | Class risk budgets | Run the existing `class_balanced` model. Check it can raise a quiet class above its raw size; if so, cap it to scale down only. Then try a per-class cap (e.g. 40% of risk). Net of costs. | Low |
| P2 | HRP weights replace inverse-vol | Correlate returns × lagged signal sign, cluster, bisect, use as relative sizes. Keep scale and trade stages unchanged. Start within class. Compare with P1. | Medium |
| P3 | Historical resampling test | Block-bootstrap inspected monthly returns; compare inverse-vol, class-balanced and HRP on variance, turnover and concentration across draws. | Medium |
| P5 | Choose by CPCV, then deflate | Pick the class cap or HRP variant with `src/backtest/cross_validation.py`; report the deflated Sharpe with the trial count. | Medium |

**Not recommended:** mean-variance or minimum-variance on this covariance (condition number, 36 months, no return forecast to optimise).

**Order:** P0, then P1, then P2 judged against P1. If class budgets give most of the gain, HRP adds complexity for little. Size discretisation (P4) is in notebook 3.

**Open questions:** (1) Hold *realised* or *forecast* volatility fixed when comparing? I assumed realised. (2) For P2, cluster on signed returns or raw returns? (3) Is a class cap a hard limit or a target?

## Tests

Logic lives in `scripts/run_pc_hold.py` (parameters in `config/pc_hold.json`) and the hold-stage pieces in `src/portfolio_construction/hold/` (`cap_class_risk`, `HrpSizing`, and a reduce-only option on `balance_classes`; `construct_targets` takes an `adjustment`). The script reshapes the sized TREND book, then runs the same limits, 10% band and 2 bps cost as the baseline, and writes to `reports/pc_hold/`. This notebook runs it if that folder is missing (a few minutes) and reads the output.

**Matched volatility.** Variants change the book's risk, so each is compared at the baseline's realised volatility: Sharpe (scale-free), and mean return, cost and drawdown multiplied by baseline volatility over variant volatility. This is a leverage-scaling proxy; it ignores that more leverage would change the gross cap and trading.

**Answers I assumed to the open questions.**
1. *Realised volatility is held fixed* (the matching above), not forecast volatility.
2. *HRP clusters on signed returns*: the covariance of each position's own return (sign-flipped for shorts, from the current month's signal). The raw-covariance version is run as a check (`hrp_within_class_raw`).
3. *A class cap is a hard limit* on the sized book each month, scaling a class down only. The gross cap and risk ceiling still apply afterwards. A cap below 1/K for K active classes is raised to 1/K.

HRP positions keep the baseline's risk: within each class (or in total for `hrp_all_markets`) they are rescaled to the sized book's forecast risk, so only the relative sizes change.

In [4]:
import json

from run_pc_hold import main

HOLD = ROOT / "reports/pc_hold_dcc"
if not HOLD.exists():
    main(HOLD, "configured")
spec = json.loads((HOLD / "trial_specification.json").read_text())
variants = pd.read_csv(HOLD / "variants.csv").set_index("variant")
print(f"{len(spec['variants'])} variants tried, including the baseline")

9 variants tried, including the baseline


### P0: diagnostics on every run

Each variant is scored on the baseline's diagnostics: effective bets (mean and minimum), the largest class share of forecast risk (mean, and the share of months it exceeds 40%), and the realised/forecast risk ratio. The trial count is recorded in `trial_specification.json`: the nine variants below are nine trials on an inspected window.

In [5]:
variants[["sharpe", "vol_ann", "matched_mean_ann", "matched_max_drawdown", "turnover_ann", "effective_bets", "effective_bets_min", "max_class_share", "months_class_over_cap", "calibration"]]

,sharpe,vol_ann,matched_mean_ann,matched_max_drawdown,turnover_ann,effective_bets,effective_bets_min,max_class_share,months_class_over_cap,calibration
variant,,,,,,,,,,
inverse_vol,1.199,0.092,0.111,-0.116,8.670,22.963,5.356,0.614,0.852,1.026
class_balanced,1.172,0.091,0.108,-0.119,11.408,15.368,3.305,0.366,0.158,1.039
class_balanced_down,1.188,0.084,0.110,-0.127,8.644,21.529,5.356,0.486,0.630,1.032
class_cap_50,1.206,0.085,0.111,-0.126,8.632,21.633,3.522,0.496,0.852,1.033
class_cap_40,1.152,0.080,0.106,-0.134,8.435,20.818,3.522,0.437,0.555,1.030
class_cap_30,1.103,0.070,0.102,-0.152,8.498,19.014,3.522,0.360,0.131,1.031
hrp_within_class,1.321,0.085,0.122,-0.110,13.964,17.202,2.788,0.615,0.846,1.029
hrp_within_class_raw,1.274,0.086,0.118,-0.118,11.824,18.339,3.243,0.615,0.846,1.025
hrp_all_markets,1.228,0.076,0.113,-0.122,16.131,6.745,1.304,0.704,0.987,0.981


- **The baseline is the most diversified book of the nine, but only just.** It has 23.0 effective bets on average. Every variant lowers that: 15.4 for the equal-budget model, 17.2 for HRP within class, 6.7 for HRP across all markets. Reshaping the relative sizes does not buy more independent bets.
- **Concentration is real.** The largest class (commodities) carries 61% of forecast risk on average, over 40% in 85% of months. Baseline Sharpe is 1.20 and the matched-volatility drawdown 11.6%.
- **Realised risk matches forecast** (ratio 1.03 in the baseline; 0.98 to 1.04 across variants). The rolling model understated risk by about 27%; DCC-GARCH does not.

### P1: class risk budgets and caps

**What we are doing.** Test whether cutting the dominant class, or giving every class an equal risk budget, improves the book. Four forms: the existing equal-budget model (`class_balanced`), the same reduce-only (`class_balanced_down`), and caps on any class's share of forecast variance at 50%, 40% and 30%.

**Inputs.** The sized TREND book and the month's covariance forecast; asset-class labels.

**Process.** Equal budget: scale each class's standalone risk to ceiling / sqrt(K). Cap: scale the class with the largest Euler share down by bisection until it is at or below the cap, repeating until all are.

**Output.** Matched-volatility Sharpe, return, drawdown, turnover and concentration.

In [6]:
pd.read_csv(HOLD / "balance_check.csv", index_col=0)

,mean_factor,share_months_enlarged
COMM,0.708,0.109
FX,1.987,0.801
FI,1.493,0.593
EQ,1.818,0.480


In [7]:
variants.loc[["inverse_vol", "class_balanced", "class_balanced_down", "class_cap_50", "class_cap_40", "class_cap_30"], ["sharpe", "vol_ann", "matched_mean_ann", "matched_max_drawdown", "turnover_ann", "matched_cost_ann", "effective_bets", "max_class_share", "months_class_over_cap"]]

,sharpe,vol_ann,matched_mean_ann,matched_max_drawdown,turnover_ann,matched_cost_ann,effective_bets,max_class_share,months_class_over_cap
variant,,,,,,,,,
inverse_vol,1.199,0.092,0.111,-0.116,8.670,0.002,22.963,0.614,0.852
class_balanced,1.172,0.091,0.108,-0.119,11.408,0.002,15.368,0.366,0.158
class_balanced_down,1.188,0.084,0.110,-0.127,8.644,0.002,21.529,0.486,0.630
class_cap_50,1.206,0.085,0.111,-0.126,8.632,0.002,21.633,0.496,0.852
class_cap_40,1.152,0.080,0.106,-0.134,8.435,0.002,20.818,0.437,0.555
class_cap_30,1.103,0.070,0.102,-0.152,8.498,0.002,19.014,0.360,0.131


- **The equal-budget model does enlarge quiet classes.** Its factor averages 2.0 for FX, 1.8 for equities and 1.5 for bonds, and above 1 in 80%, 48% and 59% of months; commodities are cut (0.71 on average). So it is not reduce-only, and I added the reduce-only form.
- **Scaling up costs and does not pay.** Equal budgets give Sharpe 1.172 (baseline 1.199), raise turnover 32% (8.7 to 11.4 times NAV) and cut effective bets to 15.4.
- **Reduce-only forms are a wash.** Sharpe is 1.188 reduce-only and 1.206 at a 50% cap, within noise of 1.199, at lower volatility (8.4% and 8.5% against 9.2%). The matched drawdown is slightly deeper (12.7% and 12.6% against 11.6%).
- **Tighter caps do worse.** Sharpe is 1.152 at 40% and 1.103 at 30%, matched drawdowns 13.4% and 15.2%. The cap trades away commodity trend return for little diversification (effective bets fall to 20.8 and 19.0).

### P2: HRP replaces inverse-volatility relative sizes

**What we are doing.** Cluster the markets by correlation, order them so similar markets sit together, and split weight top-down by cluster variance (no inversion), in place of weights proportional to 1/volatility.

**Inputs.** The month's covariance (sign-flipped by position direction), markets with a nonzero position.

**Process.** Distance = sqrt((1 - correlation) / 2), single linkage, quasi-diagonal order, recursive bisection with inverse-variance cluster variances. The positions are then rescaled to the baseline's forecast risk, within each class or in total.

**Output.** As P1, so the two can be compared on the same table.

In [8]:
variants.loc[["inverse_vol", "class_balanced_down", "class_cap_50", "hrp_within_class", "hrp_within_class_raw", "hrp_all_markets"], ["sharpe", "vol_ann", "matched_mean_ann", "matched_max_drawdown", "turnover_ann", "matched_cost_ann", "effective_bets", "effective_bets_min"]]

,sharpe,vol_ann,matched_mean_ann,matched_max_drawdown,turnover_ann,matched_cost_ann,effective_bets,effective_bets_min
variant,,,,,,,,
inverse_vol,1.199,0.092,0.111,-0.116,8.670,0.002,22.963,5.356
class_balanced_down,1.188,0.084,0.110,-0.127,8.644,0.002,21.529,5.356
class_cap_50,1.206,0.085,0.111,-0.126,8.632,0.002,21.633,3.522
hrp_within_class,1.321,0.085,0.122,-0.110,13.964,0.003,17.202,2.788
hrp_within_class_raw,1.274,0.086,0.118,-0.118,11.824,0.003,18.339,3.243
hrp_all_markets,1.228,0.076,0.113,-0.122,16.131,0.004,6.745,1.304


- **HRP within class is the one variant that beats the baseline, at a price.** Sharpe is 1.321 against 1.199 (+0.12), matched mean return 12.2% against 11.1%, and the matched drawdown is *shallower* (11.0% against 11.6%). Turnover is 14.0 times NAV against 8.7 (+61%), effective bets fall to 17.2 (minimum 2.8), and the largest class share is unchanged at 61%.
- **Across all markets it is weaker:** Sharpe 1.228, turnover 16.1, 6.7 effective bets (minimum 1.3).
- **Signed clustering helps.** The raw-covariance version scores 1.274 against 1.321 for the signed one.
- **P2 now beats P1 on Sharpe** (1.321 against at most 1.206), where under the rolling model they tied. The gain comes with more turnover and less diversification, so the next two tests decide whether it is real.

### P3: historical comparison across resampled calendars

**What we are doing.** Compare the baseline, the class forms and HRP on volatility, Sharpe, turnover and concentration across resampled development histories.

**Process.** Each month's position uses earlier data, but the strategy variants were examined on the full history. I resample the 468 months in circular 12-month blocks (2,000 draws, one calendar shared by all variants) and compute each statistic per draw. Months are resampled; weights are not re-estimated, so this measures sampling variation of the inspected backtest, not an independent holdout or a re-fit. It is not the book's Monte Carlo on simulated Gaussian data.

**Output.** Median and 90% range per variant, and the share of draws in which a variant beats the baseline on the same draw.

In [9]:
boot = pd.read_csv(HOLD / "bootstrap.csv").set_index("variant")
boot[["sharpe_median", "sharpe_low", "sharpe_high", "sharpe_better_than_baseline", "vol_median", "turnover_median", "turnover_better_than_baseline", "bets_median", "bets_better_than_baseline"]]

,sharpe_median,sharpe_low,sharpe_high,sharpe_better_than_baseline,vol_median,turnover_median,turnover_better_than_baseline,bets_median,bets_better_than_baseline
variant,,,,,,,,,
inverse_vol,1.197,0.979,1.444,0.000,0.092,8.676,0.000,22.980,0.000
class_balanced,1.174,0.964,1.407,0.310,0.090,11.432,0.000,15.373,0.000
class_balanced_down,1.185,0.977,1.428,0.358,0.084,8.657,0.559,21.524,0.000
class_cap_50,1.210,1.002,1.450,0.544,0.085,8.642,0.649,21.663,0.000
class_cap_40,1.156,0.950,1.389,0.209,0.080,8.460,0.879,20.825,0.000
class_cap_30,1.108,0.904,1.344,0.092,0.070,8.513,0.667,19.048,0.000
hrp_within_class,1.321,1.100,1.577,0.960,0.084,13.983,0.000,17.229,0.000
hrp_within_class_raw,1.274,1.041,1.535,0.906,0.086,11.835,0.000,18.369,0.000
hrp_all_markets,1.233,1.009,1.461,0.585,0.076,16.136,0.000,6.741,0.000


- **HRP within class leads.** It beats the baseline on the same draw in 96% of draws (raw covariance 91%, all markets 59%). Its 90% Sharpe range, 1.10 to 1.58, sits above the baseline's 0.98 to 1.44 but still overlaps it.
- **The class forms do not separate from the baseline:** 50% cap 54%, reduce-only equal budget 36%, original equal budget 31%, 40% cap 21%, 30% cap 9%.
- **Turnover:** HRP is worse than the baseline in every draw; the caps are lower in 56% to 88% of draws.
- **Concentration:** no variant has more effective bets than the baseline in any draw. The bootstrap resamples months with fixed weights, so it measures backtest noise, not the cost of choosing HRP.

### P5: choose by CPCV, then deflate

**What we are doing.** Select a variant on training months and score it on held-out months, then deflate the result for the number of variants tried. Some training months are later than the held-out months, so this is not a chronological or independent out-of-sample test.

**Process.** `cross_validation.py` with 8 groups, 2 held out, 36-month embargo (28 splits, 7 backtest paths). Each split picks the candidate with the best training Sharpe; all nine, including the baseline, compete. The deflated Sharpe (Bailey and Lopez de Prado) compares each series to the best expected from n skill-free trials with the same dispersion, with n = 9 (8 for the difference series) and the sensitivities 25 and 100, and is run on the returns and on the difference from the baseline.

In [10]:
splits = pd.read_csv(HOLD / "cpcv_splits.csv")
paths = pd.read_csv(HOLD / "cpcv_paths.csv")
pd.Series(
    {
        "splits": len(splits),
        "chosen variant beats baseline on held-out months": (splits.test_sharpe > splits.baseline_test_sharpe).mean(),
        "mean held-out Sharpe gap to baseline": (splits.test_sharpe - splits.baseline_test_sharpe).mean(),
        "probability of backtest overfitting (rank below median)": (splits.test_rank > (splits.candidates + 1) / 2).mean(),
        "baseline chosen": (splits.chosen == "inverse_vol").mean(),
        "path Sharpe, mean of 7": paths.sharpe.mean(),
        "path Sharpe, range low": paths.sharpe.min(),
        "path Sharpe, range high": paths.sharpe.max(),
        "baseline full-sample Sharpe": variants.loc["inverse_vol", "sharpe"],
    }
).to_frame("value")

,value
splits,28.000
chosen variant beats baseline on held-out months,0.821
mean held-out Sharpe gap to baseline,0.113
probability of backtest overfitting (rank below median),0.107
baseline chosen,0.036
"path Sharpe, mean of 7",1.303
"path Sharpe, range low",1.263
"path Sharpe, range high",1.325
baseline full-sample Sharpe,1.199


In [11]:
splits.chosen.value_counts().to_frame("splits chosen")

,splits chosen
chosen,
hrp_within_class,24
hrp_within_class_raw,3
inverse_vol,1


In [12]:
pd.read_csv(HOLD / "deflated_sharpe.csv", index_col=0)

,levels_trials_9,levels_trials_25,levels_trials_100,difference_trials_8,difference_trials_25,difference_trials_100
variant,,,,,,
class_balanced,1.000,1.000,1.000,0.000,0.000,0.000
class_balanced_down,1.000,1.000,1.000,0.000,0.000,0.000
class_cap_30,1.000,1.000,1.000,0.000,0.000,0.000
class_cap_40,1.000,1.000,1.000,0.000,0.000,0.000
class_cap_50,1.000,1.000,1.000,0.000,0.000,0.000
hrp_all_markets,1.000,1.000,1.000,0.000,0.000,0.000
hrp_within_class,1.000,1.000,1.000,0.006,0.000,0.000
hrp_within_class_raw,1.000,1.000,1.000,0.002,0.000,0.000
inverse_vol,1.000,1.000,1.000,NaN,NaN,NaN


- **The chosen variant beats the baseline in historical cross-validation.** It beats the baseline on held-out months in 82% of 28 splits, by +0.11 Sharpe on average, and the backtest paths score 1.26 to 1.32 (mean 1.30) against the baseline's 1.20. The probability of backtest overfitting is 11%. These splits are not independent out-of-sample evidence.
- **The pick is stable:** HRP within class is chosen in 24 of 28 splits, its raw-covariance form in 3 and the baseline in 1.
- **Deflation still does not clear it.** The deflated Sharpe of the levels is 1.00 for all nine, which only says the strategy is far from skill-free. On the difference from the baseline, HRP within class scores 0.006 at 8 trials and 0 at 25 and 100: a +0.12 Sharpe gain is not larger than the best of that many similar variants would show by chance on this window.

## Summary and recommendations

**What we found** (DCC-GARCH with rescale, TREND, 10% band, 2 bps, 1976-2014, in sample; nine variants tried).
- **Concentration exists:** commodities carry 61% of forecast risk on average, over 40% in 85% of months. The inverse-volatility book still has the most effective bets (23.0) of the nine.
- **P1:** the existing equal-budget model enlarges FX, bonds and equities by 1.5 to 2.0 times, adds 32% turnover and gains no Sharpe (1.172). Reduce-only forms and the 50% cap sit within noise of the baseline (1.188 and 1.206 against 1.199). Caps at 40% and 30% are worse.
- **P2:** HRP within class lifts Sharpe to 1.321, with a shallower matched drawdown (11.0%), at 61% more turnover and 17 effective bets. Across all markets it is 1.228.
- **P3 and P5:** HRP within class beats the baseline in 96% of resampled draws and in 82% of held-out CPCV splits (+0.11 Sharpe, overfitting probability 11%). The deflated difference from the baseline is still about 0.

**Recommendations**
1. **HRP within class is the leading candidate to replace inverse-volatility sizing.** Under the rolling model it tied the class caps; under DCC-GARCH, the project's configured model, it leads on Sharpe, return, drawdown, resampling and held-out selection. This reverses the rolling-model conclusion.
2. **Do not adopt it on this window alone.** The deflated difference is about 0, nine variants were tried on an inspected sample, and HRP costs 61% more turnover and 25% fewer effective bets. Confirm on data the sample has not seen (the prospective freeze) and check net-of-cost results at realistic trading costs.
3. **Do not enable the upward class balancing** (`class_balanced`). It is the costliest class option and gains nothing under either model.
4. **If a governance limit on commodity risk is wanted, use a reduce-only cap at 50%** and describe it as a risk limit: Sharpe is unchanged within noise and volatility is lower.
5. **Stop adding hold-stage variants on this window.** Each further variant raises the deflation bar.

**Limits.** In-sample window already inspected. The resampling keeps weights fixed; it measures noise in the backtest, not re-estimation. Matched volatility is a leverage-scaling proxy. HRP uses single linkage and the current month's signs; other linkages or time-varying signs were not tried. The DCC-GARCH covariance is worse conditioned (about 3,600 against about 1,200), which HRP avoids by not inverting it. The rolling-model run is kept in `reports/pc_hold/` for comparison; its recommendation was to keep inverse-volatility.